In [1]:
!pip install ultralytics
!pip install gradio
!pip install twilio
!pip install opencv-python-headless

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.3/41.3 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 58.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 66.3 MB/s eta 0:00:00


Load YOLO11n

In [2]:
from ultralytics import YOLO

model = YOLO("yolo11n.pt")

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


Tour Registration

In [3]:
import gradio as gr

def upload_image(image):
    return image

demo = gr.Interface(
    fn=upload_image,
    inputs=gr.Image(type="numpy"),
    outputs=gr.Image()
)

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://173b73a2b274edaecc.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [4]:
from ultralytics import YOLO
import gradio as gr

model = YOLO("yolo11n.pt")

def count_people(image):

    results = model(image)

    count = 0

    for box in results[0].boxes:

        cls = int(box.cls)

        if cls == 0:      # person
            count += 1

    return f"Tourists Detected: {count}"

demo = gr.Interface(
    fn=count_people,
    inputs=gr.Image(type="numpy"),
    outputs="text"
)

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://45229ac3cb82cd6802.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [5]:
# =====================================================
# INSTALL
# =====================================================
!pip install -q ultralytics gradio opencv-python-headless

# =====================================================
# IMPORTS
# =====================================================
import gradio as gr
import cv2
import numpy as np
from ultralytics import YOLO

# =====================================================
# LOAD MODEL
# =====================================================
# Better than yolo11n for crowd photos
model = YOLO("yolo11m.pt")

# =====================================================
# COUNT TOURISTS
# =====================================================
def count_tourists(image):

    image_bgr = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)

    results = model.predict(
        image_bgr,
        conf=0.15,
        imgsz=1280,
        verbose=False
    )

    result = results[0]

    person_count = 0

    for box in result.boxes:

        cls = int(box.cls[0])

        # COCO class 0 = person
        if cls == 0:
            person_count += 1

    annotated = result.plot()

    annotated = cv2.cvtColor(
        annotated,
        cv2.COLOR_BGR2RGB
    )

    return (
        annotated,
        f"Tourists Detected: {person_count}"
    )

# =====================================================
# GRADIO UI
# =====================================================
demo = gr.Interface(
    fn=count_tourists,
    inputs=gr.Image(type="numpy", label="Upload Group Photo"),
    outputs=[
        gr.Image(label="Detected Persons"),
        gr.Textbox(label="Result")
    ],
    title="TourGuard AI",
    description="AI Tourist Counting System"
)

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://45f99d246e7e3c929b.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [6]:
# =====================================================
# INSTALL
# =====================================================
!pip install -q ultralytics gradio opencv-python-headless

# =====================================================
# IMPORTS
# =====================================================
import cv2
import gradio as gr
from ultralytics import YOLO

# =====================================================
# LOAD MODEL
# =====================================================
model = YOLO("yolo11m.pt")

# =====================================================
# PERSON COUNT FUNCTION
# =====================================================
def count_people(image):

    results = model.predict(
        image,
        conf=0.15,
        imgsz=1280,
        verbose=False
    )

    count = 0

    for box in results[0].boxes:

        cls = int(box.cls[0])

        if cls == 0:      # person
            count += 1

    annotated = results[0].plot()

    annotated = cv2.cvtColor(
        annotated,
        cv2.COLOR_BGR2RGB
    )

    return count, annotated


# =====================================================
# COMPARE TOUR GROUPS
# =====================================================
def verify_group(start_image, current_image):

    expected_count, start_output = count_people(start_image)

    current_count, current_output = count_people(current_image)

    missing = expected_count - current_count

    if missing <= 0:
        status = "✅ SAFE - All Tourists Present"
    else:
        status = f"⚠ ALERT - Missing Tourists: {missing}"

    report = f"""
Expected Tourists : {expected_count}

Current Tourists : {current_count}

Missing Tourists : {max(missing,0)}

Status : {status}
"""

    return (
        start_output,
        current_output,
        report
    )


# =====================================================
# GRADIO UI
# =====================================================
demo = gr.Interface(
    fn=verify_group,
    inputs=[
        gr.Image(type="numpy", label="group_start.jpg"),
        gr.Image(type="numpy", label="current_group.jpg")
    ],
    outputs=[
        gr.Image(label="Start Group Detection"),
        gr.Image(label="Current Group Detection"),
        gr.Textbox(label="TourGuard Report")
    ],
    title="TourGuard AI",
    description="Tourist Verification System"
)

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://cc2cab00586da0f9ba.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [11]:
# =====================================================
# TOURGUARD AI
# Tourist Verification System
# Google Colab + Gradio + YOLO11m
# =====================================================

# =====================================================
# INSTALL
# =====================================================
!pip install -q ultralytics gradio opencv-python-headless

# =====================================================
# IMPORTS
# =====================================================
import cv2
import gradio as gr
from ultralytics import YOLO

# =====================================================
# LOAD MODEL
# =====================================================
model = YOLO("yolo11m.pt")

# =====================================================
# PERSON COUNT FUNCTION
# =====================================================
def count_people(image):

    results = model.predict(
        image,
        conf=0.15,
        imgsz=1280,
        verbose=False
    )

    count = 0

    for box in results[0].boxes:

        cls = int(box.cls[0])

        # COCO class 0 = person
        if cls == 0:
            count += 1

    annotated = results[0].plot()

    annotated = cv2.cvtColor(
        annotated,
        cv2.COLOR_BGR2RGB
    )

    return count, annotated

# =====================================================
# TOUR VERIFICATION LOGIC
# =====================================================
def verify_group(start_image, current_image):

    # Count persons in starting image
    expected_count, start_output = count_people(start_image)

    # Count persons in current image
    current_count, current_output = count_people(current_image)

    # Difference
    difference = current_count - expected_count

    # =================================================
    # CASE 1 : PERFECT MATCH
    # =================================================
    if difference == 0:

        status = "✅ PERFECT MATCH"

        report = f"""
Expected Tourists : {expected_count}

Current Tourists : {current_count}

Missing Tourists : 0

Extra Persons : 0

Status : {status}
"""

    # =================================================
    # CASE 2 : MISSING TOURISTS
    # =================================================
    elif difference < 0:

        missing = abs(difference)

        status = f"⚠ ALERT - Missing Tourists : {missing}"

        report = f"""
Expected Tourists : {expected_count}

Current Tourists : {current_count}

Missing Tourists : {missing}

Extra Persons : 0

Status : {status}
"""

    # =================================================
    # CASE 3 : EXTRA PERSONS
    # =================================================
    else:

        extra = difference

        status = f"⚠ ALERT - Extra Persons Detected : {extra}"

        report = f"""
Expected Tourists : {expected_count}

Current Tourists : {current_count}

Missing Tourists : 0

Extra Persons : {extra}

Status : {status}
"""

    return (
        start_output,
        current_output,
        report
    )

# =====================================================
# GRADIO INTERFACE
# =====================================================
demo = gr.Interface(
    fn=verify_group,

    inputs=[
        gr.Image(
            type="numpy",
            label="Tour Start Photo (group_start.jpg)"
        ),

        gr.Image(
            type="numpy",
            label="Current Group Photo (current_group.jpg)"
        )
    ],

    outputs=[
        gr.Image(
            label="Start Group Detection"
        ),

        gr.Image(
            label="Current Group Detection"
        ),

        gr.Textbox(
            label="TourGuard AI Report",
            lines=12

        )
    ],

    title="🚌 TourGuard AI",

    description="""
AI-Based Tourist Verification System

✔ Counts tourists in starting image
✔ Counts tourists in current image
✔ Detects missing tourists
✔ Detects extra persons
✔ Generates verification report
"""
)

# =====================================================
# LAUNCH
# =====================================================
demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://8faccc50d35ded0cf8.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


show where Gradio stored the flagged data.

In [12]:
!find . -iname "*flag*"

./.gradio/flagged


show where Gradio stored the flagged data.

In [13]:
import os

for root, dirs, files in os.walk("."):
    if "flagged" in root.lower():
        print(root)
        for f in files:
            print("  ", f)

./.gradio/flagged
   dataset1.csv
./.gradio/flagged/Start Group Detection
./.gradio/flagged/Start Group Detection/d4bee834a8235b96e003
   image.webp
./.gradio/flagged/Start Group Detection/cf25697ab5e25d16cce7
   image.webp
./.gradio/flagged/Current Group Detection
./.gradio/flagged/Current Group Detection/4e95cd806aff38ceac80
   image.webp
./.gradio/flagged/Current Group Detection/d1544f79a3f68ad2f680
   image.webp
./.gradio/flagged/Current Group Photo current_group.jpg)
./.gradio/flagged/Current Group Photo current_group.jpg)/633aefe315e04fc0db0d
   current_group.png
./.gradio/flagged/Current Group Photo current_group.jpg)/74249f89eab05c738dd9
   current_group.png
./.gradio/flagged/Tour Start Photo group_start.jpg)
./.gradio/flagged/Tour Start Photo group_start.jpg)/a5d22e8777376bfd16ec
   group_start.png
./.gradio/flagged/Tour Start Photo group_start.jpg)/84b51071347c9f006198
   group_start.png


First check the exact file path:

In [17]:
!find . -name "*.csv"

./.gradio/flagged/dataset1.csv
./sample_data/mnist_train_small.csv
./sample_data/california_housing_test.csv
./sample_data/california_housing_train.csv
./sample_data/mnist_test.csv


To see all columns:

In [21]:
import pandas as pd

df = pd.read_csv("./.gradio/flagged/dataset1.csv")

print(df)

                  Tour Start Photo (group_start.jpg)  ...                   timestamp
0  .gradio/flagged/Tour Start Photo group_start.j...  ...  2026-06-07 13:02:41.783893
1  .gradio/flagged/Tour Start Photo group_start.j...  ...  2026-06-07 13:05:18.652300

[2 rows x 6 columns]


To see all the columns and complete content, run:

In [22]:
import pandas as pd

df = pd.read_csv("./.gradio/flagged/dataset1.csv")

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)

print(df)

                                                       Tour Start Photo (group_start.jpg)  \
0  .gradio/flagged/Tour Start Photo group_start.jpg)/a5d22e8777376bfd16ec/group_start.png   
1  .gradio/flagged/Tour Start Photo group_start.jpg)/84b51071347c9f006198/group_start.png   

                                                         Current Group Photo (current_group.jpg)  \
0  .gradio/flagged/Current Group Photo current_group.jpg)/633aefe315e04fc0db0d/current_group.png   
1  .gradio/flagged/Current Group Photo current_group.jpg)/74249f89eab05c738dd9/current_group.png   

                                                   Start Group Detection  \
0  .gradio/flagged/Start Group Detection/d4bee834a8235b96e003/image.webp   
1  .gradio/flagged/Start Group Detection/cf25697ab5e25d16cce7/image.webp   

                                                   Current Group Detection  \
0  .gradio/flagged/Current Group Detection/4e95cd806aff38ceac80/image.webp   
1  .gradio/flagged/Current Group D

To see only the column names:

In [23]:
print(df.columns)

Index(['Tour Start Photo (group_start.jpg)',
       'Current Group Photo (current_group.jpg)', 'Start Group Detection',
       'Current Group Detection', 'TourGuard AI Report', 'timestamp'],
      dtype='object')


To see the TourGuard report that was saved

In [24]:
print(df.iloc[0])

Tour Start Photo (group_start.jpg)                                                                     .gradio/flagged/Tour Start Photo group_start.jpg)/a5d22e8777376bfd16ec/group_start.png
Current Group Photo (current_group.jpg)                                                         .gradio/flagged/Current Group Photo current_group.jpg)/633aefe315e04fc0db0d/current_group.png
Start Group Detection                                                                                                   .gradio/flagged/Start Group Detection/d4bee834a8235b96e003/image.webp
Current Group Detection                                                                                               .gradio/flagged/Current Group Detection/4e95cd806aff38ceac80/image.webp
TourGuard AI Report                        '\nExpected Tourists : 51\n\nCurrent Tourists : 56\n\nMissing Tourists : 0\n\nExtra Persons : 5\n\nStatus : ⚠ ALERT - Extra Persons Detected : 5\n
timestamp                                         

To see the second flagged record:

In [30]:
print(df.iloc[1])

Tour Start Photo (group_start.jpg)                                                                     .gradio/flagged/Tour Start Photo group_start.jpg)/84b51071347c9f006198/group_start.png
Current Group Photo (current_group.jpg)                                                         .gradio/flagged/Current Group Photo current_group.jpg)/74249f89eab05c738dd9/current_group.png
Start Group Detection                                                                                                   .gradio/flagged/Start Group Detection/cf25697ab5e25d16cce7/image.webp
Current Group Detection                                                                                               .gradio/flagged/Current Group Detection/d1544f79a3f68ad2f680/image.webp
TourGuard AI Report                        '\nExpected Tourists : 51\n\nCurrent Tourists : 56\n\nMissing Tourists : 0\n\nExtra Persons : 5\n\nStatus : ⚠ ALERT - Extra Persons Detected : 5\n
timestamp                                         

To see the TourGuard report that was saved

In [26]:
print(df["TourGuard AI Report"])

0    '\nExpected Tourists : 51\n\nCurrent Tourists : 56\n\nMissing Tourists : 0\n\nExtra Persons : 5\n\nStatus : ⚠ ALERT - Extra Persons Detected : 5\n
1    '\nExpected Tourists : 51\n\nCurrent Tourists : 56\n\nMissing Tourists : 0\n\nExtra Persons : 5\n\nStatus : ⚠ ALERT - Extra Persons Detected : 5\n
Name: TourGuard AI Report, dtype: object


To verify its exact location, run

In [27]:
!find . -name "dataset1.csv"

./.gradio/flagged/dataset1.csv


Download the CSV

In [29]:
from google.colab import files

files.download("./.gradio/flagged/dataset1.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>